# Week1_ex1 — Two-Bus System with an Isolated Bus

Reproduces Examples 1.1 and 1.2 from Glover, Overbye & Sarma, *Power System Analysis and Design* (7th ed.). Example 1.2 is Example 1.1 with an extra bus left unconnected, so one network covers both and shows how PowerWorld and pandapower report a dead bus.

![PowerWorld one-line](Images/Week1_ex1_result_image.png)

In [1]:
import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.topology as top

# 1. Per-unit bases used by the PowerWorld case
S_BASE_MVA = 100.0                      # back-calculated: I = 0.05 pu, I^2 R = 0.00075 pu = 0.075 MW only on 100 MVA
V_BASE_KV = 16.0                        # nominal voltage of every bus
Z_BASE_OHM = V_BASE_KV**2 / S_BASE_MVA  # 2.56 ohm

# 2. Line data read once from PowerWorld Simulator (Run Mode, Model Explorer > Network > Branches Input)
R_PU, X_PU, B_PU = 0.30, 0.60, 0.0
LIM_MVA = 20.0

# 3. Convert per-unit impedance to pandapower inputs (length fixed at 1 km, so ohm/km = ohm)
r_ohm = R_PU * Z_BASE_OHM
x_ohm = X_PU * Z_BASE_OHM
max_i_ka = LIM_MVA / (np.sqrt(3) * V_BASE_KV)   # MVA rating -> current rating at nominal voltage

pd.DataFrame({
    "Quantity": ["Z_base [ohm]", "R [ohm]", "X [ohm]", "I_max [kA]"],
    "Value": [Z_BASE_OHM, r_ohm, x_ohm, max_i_ka],
}).round(4)

,Quantity,Value
0,Z_base [ohm],2.5600
1,R [ohm],0.7680
2,X [ohm],1.5360
3,I_max [kA],0.7217


In [2]:
# 4. Build the network: slack at Bus 1, one line to the 5 MW load at Bus 2, Bus 3 left floating
net = pp.create_empty_network(f_hz=60, sn_mva=S_BASE_MVA)
b1 = pp.create_bus(net, V_BASE_KV, name="Bus 1")
b2 = pp.create_bus(net, V_BASE_KV, name="Bus 2")
b3 = pp.create_bus(net, V_BASE_KV, name="Bus 3")
pp.create_ext_grid(net, b1, vm_pu=1.0)
pp.create_line_from_parameters(net, b1, b2, length_km=1.0, r_ohm_per_km=r_ohm,
                               x_ohm_per_km=x_ohm, c_nf_per_km=0.0, max_i_ka=max_i_ka)
pp.create_load(net, b2, p_mw=5.0, q_mvar=0.0)

# 5. Solve
pp.runpp(net, numba=False)
pd.DataFrame({
    "Bus": net.bus.name,
    "Voltage [pu]": net.res_bus.vm_pu,
    "Angle [deg]": net.res_bus.va_degree,
}).round(5)

,Bus,Voltage [pu],Angle [deg]
0,Bus 1,1.0000,0.00000
1,Bus 2,0.9843,-1.74657
2,Bus 3,NaN,NaN


In [3]:
# 6. Reference values read once from PowerWorld Simulator (Run Mode, Model Explorer > Buses / Branches State)
pw = {
    "Bus 2 voltage [pu]": 0.98430,
    "Bus 2 angle [deg]": -1.75,
    "Slack P [MW]": 5.08,
    "Slack Q [Mvar]": 0.15,
    "Line P loss [MW]": 0.08,
    "Line Q loss [Mvar]": 0.15,
    "Line loading [%]": 25.4,
}
ppw = {
    "Bus 2 voltage [pu]": net.res_bus.vm_pu.at[b2],
    "Bus 2 angle [deg]": net.res_bus.va_degree.at[b2],
    "Slack P [MW]": net.res_ext_grid.p_mw.iat[0],
    "Slack Q [Mvar]": net.res_ext_grid.q_mvar.iat[0],
    "Line P loss [MW]": net.res_line.pl_mw.iat[0],
    "Line Q loss [Mvar]": net.res_line.ql_mvar.iat[0],
    "Line loading [%]": net.res_line.loading_percent.iat[0],
}
cmp = pd.DataFrame({"PowerWorld": pw, "pandapower": ppw})
cmp["Diff"] = cmp["pandapower"] - cmp["PowerWorld"]

# 7. PowerWorld rounds to its display precision, so allow half of the last shown digit
cmp["Tolerance"] = [5e-6, 5e-3, 5e-3, 5e-3, 5e-3, 5e-3, 5e-2]
assert (cmp["Diff"].abs() <= cmp["Tolerance"]).all()
cmp.round(5)

,PowerWorld,pandapower,Diff,Tolerance
Bus 2 voltage [pu],0.9843,0.98430,-0.00000,0.000
Bus 2 angle [deg],-1.7500,-1.74657,0.00343,0.005
Slack P [MW],5.0800,5.07741,-0.00259,0.005
Slack Q [Mvar],0.1500,0.15482,0.00482,0.005
Line P loss [MW],0.0800,0.07741,-0.00259,0.005
Line Q loss [Mvar],0.1500,0.15482,0.00482,0.005
Line loading [%],25.4000,25.39886,-0.00114,0.050


In [4]:
# 8. Bus 3: PowerWorld marks it Dead and stores 0.01 pu (one-line label still shows 0.00 kV);
#    pandapower drops it from the internal Ybus and returns NaN
pw_dead_vm = 0.01                                   # PowerWorld Buses table, Bus 3 row (greyed out)
ybus = net._ppc["internal"]["Ybus"].toarray()

pd.DataFrame({
    "Item": ["Bus 3 voltage [pu]", "Ybus size", "Bus 3 status", "Unsupplied buses"],
    "PowerWorld": [pw_dead_vm, "3 x 3 (zero row/col)", "Dead", "-"],
    "pandapower": [net.res_bus.vm_pu.at[b3], f"{ybus.shape[0]} x {ybus.shape[1]}",
                   "in service, result NaN", str(sorted(top.unsupplied_buses(net)))],
})

,Item,PowerWorld,pandapower
0,Bus 3 voltage [pu],0.01,NaN
1,Ybus size,3 x 3 (zero row/col),2 x 2
2,Bus 3 status,Dead,"in service, result NaN"
3,Unsupplied buses,-,[2]


In [5]:
# 9. Pitfall: the "worst bus voltage" KPI depends on how the dead bus is stored
vm_pw_style = net.res_bus.vm_pu.fillna(pw_dead_vm)   # what you get if PowerWorld values are pasted in
supplied = ~net.bus.index.isin(top.unsupplied_buses(net))

pd.DataFrame({
    "Method": ["pandapower min (NaN skipped)", "PowerWorld-style min (0.01 kept)", "Min over supplied buses only"],
    "Min voltage [pu]": [net.res_bus.vm_pu.min(), vm_pw_style.min(), vm_pw_style[supplied].min()],
}).round(5)

,Method,Min voltage [pu]
0,pandapower min (NaN skipped),0.9843
1,PowerWorld-style min (0.01 kept),0.0100
2,Min over supplied buses only,0.9843
